## LightGBM Baseline for BAF Variants

This notebook runs a reviewer-safe LightGBM baseline under the same protocol used by the current FT/SNN experiments:

- same BAF temporal split (`month <= 5` for train/valid, `month >= 6` for test)
- same `VALID` threshold selection under an `FPR <= 5%` constraint
- same `TEST` reporting split
- same paper-artifact export pattern for tables, curves, predictions, and manifests

Environment variables supported:

- `EXPERIMENT_TAG` (`Base`, `VI`, `VII`, `VIII`, `VIV`, `VV`)
- `BAF_DATASET_PATH` or `BAF_DATASET_DIR`
- `LIGHTGBM_N_JOBS`
- `LIGHTGBM_DEVICE_TYPE` (`cpu`, `gpu`, or `cuda`)
- `LIGHTGBM_GPU_PLATFORM_ID` / `LIGHTGBM_GPU_DEVICE_ID`
- `TUNING_TRIALS`


In [ ]:
import os, sys, json, csv, time, subprocess, warnings, re, math
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    precision_recall_curve,
    auc,
)
from sklearn.model_selection import train_test_split as sklearn_train_test_split
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

try:
    display  # type: ignore[name-defined]
except NameError:
    def display(value):
        print(value)

def _parse_env_int(env_name, default_value):
    raw_value = os.environ.get(env_name, '').strip()
    if not raw_value:
        return int(default_value)
    try:
        return int(raw_value)
    except Exception:
        print(f"Warning: invalid integer for {env_name}='{raw_value}', using default={default_value}")
        return int(default_value)

RANDOM_SEED = _parse_env_int('RANDOM_SEED', 42)
TUNING_TRIALS = _parse_env_int('TUNING_TRIALS', 100)
LIGHTGBM_N_JOBS = _parse_env_int('LIGHTGBM_N_JOBS', max(1, os.cpu_count() or 1))
LIGHTGBM_DEVICE_TYPE = os.environ.get('LIGHTGBM_DEVICE_TYPE', 'cpu').strip().lower() or 'cpu'
LIGHTGBM_GPU_PLATFORM_ID = _parse_env_int('LIGHTGBM_GPU_PLATFORM_ID', 0)
LIGHTGBM_GPU_DEVICE_ID = _parse_env_int('LIGHTGBM_GPU_DEVICE_ID', 0)

AUTO_INSTALL_PACKAGES = (
    'lightgbm',
    'optuna',
    'tqdm',
    'matplotlib',
    'scikit-learn',
    'pandas',
    'numpy',
    'aequitas',
    'seaborn',
    'ipywidgets',
)

EXPERIMENT_TAG = os.environ.get('EXPERIMENT_TAG', 'Base').strip() or 'Base'
RESULTS_DIR = Path('.')
DATASET_DIR = Path('baf-datasets')
DATASET_FILE_NAME = f'{EXPERIMENT_TAG}.csv'
DATA_CSV_PATH = DATASET_DIR / DATASET_FILE_NAME

RUN_OUTPUT_ROOT = RESULTS_DIR / f'lightgbm_baseline_100_run_{EXPERIMENT_TAG}'
OPTUNA_DIR = RUN_OUTPUT_ROOT / 'optuna'
CSV_EXPORT_DIR = RUN_OUTPUT_ROOT / 'csv_exports'
MODEL_EXPORT_DIR = RUN_OUTPUT_ROOT / 'model_artifacts'
PAPER_EXPORT_DIR = RUN_OUTPUT_ROOT / 'paper_artifacts'

TARGET_CANDIDATE_COLUMNS = ['fraud_bool', 'fraud', 'is_fraud', 'label', 'target']
MONTH_COLUMN = 'month'
TRAIN_MONTH_MAX = 5
TEST_MONTH_MIN = 6
VALID_SPLIT_RATIO = 0.2
DROP_FEATURE_NAME_FRAGMENTS = {'customer', 'id', 'uuid'}
INT_AS_CATEGORICAL_MAX_UNIQUE = 50

AGE_GROUP_COLUMNS = ['age', 'Age', 'AGE', 'age_group', 'customer_age']
AGE_THRESHOLD = 50
INCOME_GROUP_COLUMNS = ['income', 'Income', 'annual_income', 'AMT_INCOME_TOTAL']
EMPLOYMENT_STATUS_GROUP_COLUMNS = ['employment_status', 'EmploymentStatus', 'NAME_INCOME_TYPE']
INCOME_GROUP_QUANTILES = [0.0, 0.25, 0.5, 0.75, 1.0]
AEQUITAS_FAIRNESS_ATTRIBUTES = ['age_group', 'income_group', 'employment_status_group']

EPSILON = 1e-12
DEFAULT_THRESHOLD = 0.5
FPR_CAP = 0.05
SECONDARY_FPR_CAP = 0.01
LOG_TEST_DURING_TUNING = True
RESET_TRIAL_LOG_CSVS = False

MAX_ESTIMATORS = 5000
EARLY_STOPPING_ROUNDS = 200
OPTUNA_DIRECTION = 'maximize'
OPTUNA_STUDY_NAME = f'lightgbm_baseline_recall_at_fpr_cap_{EXPERIMENT_TAG}'
OPTUNA_STORAGE_PATH = OPTUNA_DIR / f'optuna_{OPTUNA_STUDY_NAME}.db'
OPTUNA_STORAGE_URL = f'sqlite:///{OPTUNA_STORAGE_PATH.resolve()}'

TRIALS_CSV = CSV_EXPORT_DIR / f'lightgbm_baseline_100_trials_{EXPERIMENT_TAG}.csv'
BEST_CSV = CSV_EXPORT_DIR / f'lightgbm_baseline_100_trials_best_{EXPERIMENT_TAG}.csv'
FAIR_CSV = CSV_EXPORT_DIR / f'lightgbm_baseline_100_trials_fairness_{EXPERIMENT_TAG}.csv'

TRIAL_LOG_FIELDS = [
    'stage', 'trial', 'best_iteration',
    'boosting_type', 'num_leaves', 'max_depth', 'learning_rate', 'min_child_samples',
    'subsample', 'subsample_freq', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'min_split_gain',
    'val_auc', 'val_prauc', 'val_recall_at_fpr', 'val_fpr', 'val_threshold'
]
FAIR_LOG_FIELDS = [
    'stage', 'trial', 'best_iteration',
    'perf_recall_test', 'fairness_ratio_test', 'predictive_equality_age_test', 'thr_star', 'fpr_test',
    'recall_val', 'fpr_val', 'fairness_ratio_val', 'predictive_equality_age_val',
    'aeq_fpr_parity_age_test', 'aeq_fpr_parity_income_test', 'aeq_fpr_parity_employment_status_test',
    'aeq_fpr_parity_age_val', 'aeq_fpr_parity_income_val', 'aeq_fpr_parity_employment_status_val',
    'attribute_n_distinct_groups', 'attribute_is_degenerate'
]

PLOT_FIGSIZE = (7.0, 5.0)
PLOT_DPI = 150
TOP_K_TRIALS = 5


In [ ]:
PACKAGE_IMPORT_NAME_MAP = {
    'scikit-learn': 'sklearn',
}
OPTIONAL_PIP_PACKAGES = {'ipywidgets'}
optional_install_failures = {}


def ensure_typing_extensions_features():
    required_attributes = ('TypeIs', 'TypeAliasType')
    try:
        import typing_extensions as typing_extensions_module
    except Exception:
        typing_extensions_module = None

    has_required_attributes = (
        typing_extensions_module is not None
        and all(hasattr(typing_extensions_module, attr_name) for attr_name in required_attributes)
    )
    if has_required_attributes:
        return

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'typing_extensions'])
    import importlib
    typing_extensions_module = importlib.import_module('typing_extensions')
    typing_extensions_module = importlib.reload(typing_extensions_module)

    missing_attributes = [
        attr_name for attr_name in required_attributes if not hasattr(typing_extensions_module, attr_name)
    ]
    if missing_attributes:
        raise ImportError(
            'typing_extensions is missing required attributes after upgrade: ' + ', '.join(missing_attributes)
        )
    print('[deps] upgraded typing_extensions for aequitas compatibility.')


def import_lightgbm_without_dask():
    import builtins
    import importlib

    real_import = builtins.__import__

    def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
        if name == 'dask' or name.startswith('dask.'):
            raise ImportError('Blocked optional dask import for lightgbm compatibility')
        return real_import(name, globals, locals, fromlist, level)

    builtins.__import__ = guarded_import
    try:
        return importlib.import_module('lightgbm')
    finally:
        builtins.__import__ = real_import


ensure_typing_extensions_features()

for package_name in AUTO_INSTALL_PACKAGES:
    import_name = PACKAGE_IMPORT_NAME_MAP.get(package_name, package_name)
    try:
        if package_name == 'lightgbm':
            import_lightgbm_without_dask()
        else:
            __import__(import_name)
        continue
    except Exception:
        pass

    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        if package_name == 'lightgbm':
            import_lightgbm_without_dask()
        else:
            __import__(import_name)
        print(f'[deps] installed: {package_name}')
    except Exception as install_error:
        if package_name in OPTIONAL_PIP_PACKAGES:
            optional_install_failures[package_name] = str(install_error)
            print(f'[deps] optional package unavailable, continuing: {package_name} | {install_error}')
        else:
            raise

if optional_install_failures:
    print('Optional dependency install failures:')
    for package_name, failure_text in optional_install_failures.items():
        print(f' - {package_name}: {failure_text}')

import optuna
lgb = import_lightgbm_without_dask()
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    from aequitas.group import Group as AequitasGroup
    from aequitas.bias import Bias as AequitasBias
except Exception as aequitas_import_error:
    AequitasGroup = None
    AequitasBias = None
    _AEQUITAS_IMPORT_ERROR = aequitas_import_error
else:
    _AEQUITAS_IMPORT_ERROR = None

for _path_dir in [RUN_OUTPUT_ROOT, OPTUNA_DIR, CSV_EXPORT_DIR, MODEL_EXPORT_DIR, PAPER_EXPORT_DIR]:
    _path_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
print('Using LightGBM baseline (same split/threshold protocol as current experiments).')
print(f'Experiment tag : {EXPERIMENT_TAG}')
print(f'Dataset path   : {DATA_CSV_PATH}')
print(f'Run root       : {RUN_OUTPUT_ROOT}')
print(f'Optuna DB      : {OPTUNA_STORAGE_PATH}')
print(f'Tuning budget  : {TUNING_TRIALS} trials | max_estimators={MAX_ESTIMATORS} | early_stopping_rounds={EARLY_STOPPING_ROUNDS}')
print(f'LightGBM device: {LIGHTGBM_DEVICE_TYPE}')
print(f'CPU threads    : {LIGHTGBM_N_JOBS}')
if LIGHTGBM_DEVICE_TYPE != 'cpu':
    print(f'GPU platform/id: {LIGHTGBM_GPU_PLATFORM_ID}/{LIGHTGBM_GPU_DEVICE_ID}')


In [ ]:
def resolve_dataset_path(raw_path: Path) -> Path:
    dataset_file_name = Path(DATASET_FILE_NAME).name

    env_dataset_path = os.environ.get('BAF_DATASET_PATH', '').strip()
    env_dataset_dir = os.environ.get('BAF_DATASET_DIR', '').strip()

    candidate_paths = [
        Path(raw_path),
        Path(dataset_file_name),
        Path(DATASET_DIR) / dataset_file_name,
    ]
    if env_dataset_path:
        candidate_paths.append(Path(env_dataset_path))
    if env_dataset_dir:
        candidate_paths.append(Path(env_dataset_dir) / dataset_file_name)
    candidate_paths.extend([
        Path('/content') / dataset_file_name,
        Path('/content/baf-datasets') / dataset_file_name,
        Path('/content/drive/MyDrive/baf-datasets') / dataset_file_name,
        Path('/content/drive/MyDrive/Colab Notebooks/baf-datasets') / dataset_file_name,
    ])

    seen = set()
    ordered_candidates = []
    for candidate_path in candidate_paths:
        candidate_path = Path(candidate_path)
        candidate_key = str(candidate_path)
        if candidate_key in seen:
            continue
        seen.add(candidate_key)
        ordered_candidates.append(candidate_path)

    for candidate_path in ordered_candidates:
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(
        'Dataset not found. Tried paths:\n - ' + '\n - '.join(str(path) for path in ordered_candidates)
    )


def parse_int_from_value(value):
    try:
        return int(re.findall(r'-?\d+', str(value))[0])
    except Exception:
        return np.nan


DATA_CSV_PATH = resolve_dataset_path(DATA_CSV_PATH)
DATASET_DIR = DATA_CSV_PATH.parent
full_df = pd.read_csv(DATA_CSV_PATH)
print(f'Resolved dataset path: {DATA_CSV_PATH}')

for candidate_target_col in TARGET_CANDIDATE_COLUMNS:
    if candidate_target_col in full_df.columns:
        target_col = candidate_target_col
        break
else:
    raise ValueError(f'Set target column name. Tried: {TARGET_CANDIDATE_COLUMNS}')

if MONTH_COLUMN not in full_df.columns:
    raise ValueError(f"Expected a '{MONTH_COLUMN}' column for time-based split.")

month_values = full_df[MONTH_COLUMN].map(parse_int_from_value)
if month_values.isna().any():
    raise ValueError(f"Could not parse some '{MONTH_COLUMN}' values as integers.")
full_df = full_df.assign(**{MONTH_COLUMN: month_values.astype(int)})

train_valid_df = full_df[full_df[MONTH_COLUMN] <= TRAIN_MONTH_MAX].copy()
test_df = full_df[full_df[MONTH_COLUMN] >= TEST_MONTH_MIN].copy()
if len(train_valid_df) == 0 or len(test_df) == 0:
    raise ValueError('Time split produced empty train/valid or test set.')

feature_cols = [
    column_name
    for column_name in full_df.columns
    if column_name != target_col
    and column_name != MONTH_COLUMN
    and not any(fragment in column_name.lower() for fragment in DROP_FEATURE_NAME_FRAGMENTS)
]

categorical_cols = [
    column_name for column_name in feature_cols
    if str(full_df[column_name].dtype) in ('object', 'category')
]
for column_name in feature_cols:
    if (
        column_name not in categorical_cols
        and pd.api.types.is_integer_dtype(full_df[column_name])
        and full_df[column_name].nunique() <= INT_AS_CATEGORICAL_MAX_UNIQUE
    ):
        categorical_cols.append(column_name)

continuous_cols = [
    column_name
    for column_name in feature_cols
    if column_name not in categorical_cols and pd.api.types.is_numeric_dtype(full_df[column_name])
]

print(f'Target column        : {target_col}')
print(f'Train/valid rows     : {len(train_valid_df)}')
print(f'Test rows            : {len(test_df)}')
print(f'Features total       : {len(feature_cols)}')
print(f'Categorical features : {len(categorical_cols)}')
print(f'Continuous features  : {len(continuous_cols)}')


In [ ]:
def _first_existing_column(dataframe: pd.DataFrame, candidate_columns):
    for column_name in candidate_columns:
        if column_name in dataframe.columns:
            return column_name
    return None

def extract_numeric_age(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    numeric_text = series.astype(str).str.extract(r'(-?\d+)', expand=False)
    return pd.to_numeric(numeric_text, errors='coerce')

def extract_numeric_feature(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    numeric_text = (
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.extract(r'(-?\d+\.?\d*)', expand=False)
    )
    return pd.to_numeric(numeric_text, errors='coerce')

def make_age_group_values(dataframe: pd.DataFrame):
    age_column_name = _first_existing_column(dataframe, AGE_GROUP_COLUMNS)
    if age_column_name is None:
        return pd.Series('Unknown', index=dataframe.index, dtype=object)

    parsed_age = extract_numeric_age(dataframe[age_column_name])
    age_labels = pd.Series(
        np.where(parsed_age >= AGE_THRESHOLD, f'>={AGE_THRESHOLD}', f'<{AGE_THRESHOLD}'),
        index=dataframe.index,
    )
    age_labels[parsed_age.isna()] = 'Unknown'
    return age_labels.astype(str)

def fit_income_group_bin_edges(train_dataframe: pd.DataFrame):
    income_column_name = _first_existing_column(train_dataframe, INCOME_GROUP_COLUMNS)
    if income_column_name is None:
        return None

    parsed_income = extract_numeric_feature(train_dataframe[income_column_name]).dropna()
    if parsed_income.empty or parsed_income.nunique() < 4:
        return None

    quantiles = parsed_income.quantile(INCOME_GROUP_QUANTILES).to_numpy(dtype=float)
    bin_edges = np.unique(quantiles)
    return bin_edges if len(bin_edges) >= 2 else None

def make_income_group_values(dataframe: pd.DataFrame, income_bin_edges=None):
    income_column_name = _first_existing_column(dataframe, INCOME_GROUP_COLUMNS)
    if income_column_name is None:
        return pd.Series('Unknown', index=dataframe.index, dtype=object)

    parsed_income = extract_numeric_feature(dataframe[income_column_name])
    if income_bin_edges is not None:
        bin_edges = np.unique(np.asarray(income_bin_edges, dtype=float))
        if len(bin_edges) >= 2:
            income_bins = pd.cut(parsed_income, bins=bin_edges, include_lowest=True, duplicates='drop')
            income_labels = income_bins.astype(str).replace('nan', 'Unknown')
            return income_labels.fillna('Unknown').astype(str)

    income_labels = dataframe[income_column_name].astype(str).str.strip()
    income_labels = income_labels.replace({'': 'Unknown', 'nan': 'Unknown', 'None': 'Unknown'})
    return income_labels.fillna('Unknown').astype(str)

def make_employment_status_group_values(dataframe: pd.DataFrame):
    employment_column_name = _first_existing_column(dataframe, EMPLOYMENT_STATUS_GROUP_COLUMNS)
    if employment_column_name is None:
        return pd.Series('Unknown', index=dataframe.index, dtype=object)

    employment_labels = dataframe[employment_column_name].astype(str).str.strip()
    employment_labels = employment_labels.replace({'': 'Unknown', 'nan': 'Unknown', 'None': 'Unknown'})
    return employment_labels.fillna('Unknown').astype(str)

def build_sensitive_attribute_frame(dataframe: pd.DataFrame, income_bin_edges=None):
    return pd.DataFrame(
        {
            'age_group': make_age_group_values(dataframe),
            'income_group': make_income_group_values(dataframe, income_bin_edges=income_group_bin_edges),
            'employment_status_group': make_employment_status_group_values(dataframe),
        },
        index=dataframe.index,
    )

train_df, valid_df = sklearn_train_test_split(
    train_valid_df,
    test_size=VALID_SPLIT_RATIO,
    stratify=train_valid_df[target_col],
    random_state=RANDOM_SEED,
)

income_group_bin_edges = fit_income_group_bin_edges(train_df)
train_sensitive_attributes_df = build_sensitive_attribute_frame(train_df, income_bin_edges=income_group_bin_edges)
valid_sensitive_attributes_df = build_sensitive_attribute_frame(valid_df, income_bin_edges=income_group_bin_edges)
test_sensitive_attributes_df = build_sensitive_attribute_frame(test_df, income_bin_edges=income_group_bin_edges)

def normalize_categorical_value(value):
    if pd.isna(value):
        return '__MISSING__'
    text = str(value).strip()
    if text in {'', 'nan', 'None'}:
        return '__MISSING__'
    return text

categorical_levels = {}
for column_name in categorical_cols:
    seen_levels = []
    seen_keys = set()
    for raw_value in train_valid_df[column_name].tolist():
        normalized = normalize_categorical_value(raw_value)
        if normalized in seen_keys:
            continue
        seen_keys.add(normalized)
        seen_levels.append(normalized)
    if '__UNK__' not in seen_keys:
        seen_levels.append('__UNK__')
    categorical_levels[column_name] = seen_levels

continuous_train_mean = train_valid_df[continuous_cols].astype(float).mean() if continuous_cols else None

def build_feature_frame(dataframe_part: pd.DataFrame) -> pd.DataFrame:
    feature_df = dataframe_part[feature_cols].copy()

    for column_name in continuous_cols:
        feature_df[column_name] = pd.to_numeric(feature_df[column_name], errors='coerce')
    if continuous_cols:
        feature_df[continuous_cols] = feature_df[continuous_cols].fillna(continuous_train_mean)

    for column_name in categorical_cols:
        levels = categorical_levels[column_name]
        known_values = set(levels[:-1])
        normalized_values = feature_df[column_name].map(normalize_categorical_value)
        normalized_values = normalized_values.where(normalized_values.isin(known_values), '__UNK__')
        feature_df[column_name] = pd.Categorical(normalized_values, categories=levels)

    return feature_df

X_train = build_feature_frame(train_df)
X_valid = build_feature_frame(valid_df)
X_test = build_feature_frame(test_df)
y_train_np = train_df[target_col].to_numpy(dtype=int)
y_valid_np = valid_df[target_col].to_numpy(dtype=int)
y_test_np = test_df[target_col].to_numpy(dtype=int)

train_positive = int(y_train_np.sum())
train_negative = int((1 - y_train_np).sum())
scale_pos_weight = float(train_negative / max(train_positive, 1))

print(f'Train rows            : {len(train_df)}')
print(f'Valid rows            : {len(valid_df)}')
print(f'Test rows             : {len(test_df)}')
print(f'Train positive rate   : {float(np.mean(y_train_np)):.6f}')
print(f'Train scale_pos_weight: {scale_pos_weight:.4f}')
display(train_df.head(2))


In [ ]:
def safe_div(num, den):
    den = float(den)
    return float(num) / den if den != 0 else float('nan')

def safe_binary_metric(metric_fn, y_true, y_probabilities):
    try:
        return float(metric_fn(y_true, y_probabilities))
    except ValueError:
        return float('nan')

def recall_at_fpr_cap_from_scores(y_true, y_probabilities, cap=FPR_CAP):
    y_true_np = np.asarray(y_true).astype(int).ravel()
    y_prob_np = np.asarray(y_probabilities, dtype=float).ravel()
    fpr_values, tpr_values, thresholds = roc_curve(y_true_np, y_prob_np)
    under_cap_mask = fpr_values <= float(cap)
    if not np.any(under_cap_mask):
        return 0.0, DEFAULT_THRESHOLD, 1.0

    candidate_tprs = tpr_values[under_cap_mask]
    candidate_thresholds = thresholds[under_cap_mask]
    candidate_fprs = fpr_values[under_cap_mask]
    best_index = int(np.argmax(candidate_tprs))
    return (
        float(candidate_tprs[best_index]),
        float(candidate_thresholds[best_index]),
        float(candidate_fprs[best_index]),
    )

def dylan_snippet_pr_score_from_point(precision, recall):
    try:
        precision = float(precision)
        recall = float(recall)
    except Exception:
        return float('nan')
    if not (np.isfinite(precision) and np.isfinite(recall)):
        return float('nan')
    precision = float(np.clip(precision, 0.0, 1.0))
    recall = float(np.clip(recall, 0.0, 1.0))
    return float(auc([0.0, recall, 1.0], [0.0, precision, 1.0]))

def trapezoidal_pr_auc_from_scores(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(recall[::-1], precision[::-1]))

def confusion_metrics_from_probs(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    fpr = safe_div(fp, fp + tn)
    specificity = safe_div(tn, tn + fp)
    npv = safe_div(tn, tn + fn)
    f1 = safe_div(2 * precision * recall, precision + recall) if np.isfinite(precision) and np.isfinite(recall) else float('nan')
    acc = safe_div(tp + tn, tp + tn + fp + fn)
    bal_acc = np.nanmean([recall, specificity])
    return {
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'precision': float(precision) if np.isfinite(precision) else float('nan'),
        'recall': float(recall) if np.isfinite(recall) else float('nan'),
        'fpr': float(fpr) if np.isfinite(fpr) else float('nan'),
        'specificity': float(specificity) if np.isfinite(specificity) else float('nan'),
        'npv': float(npv) if np.isfinite(npv) else float('nan'),
        'f1': float(f1) if np.isfinite(f1) else float('nan'),
        'accuracy': float(acc) if np.isfinite(acc) else float('nan'),
        'balanced_accuracy': float(bal_acc) if np.isfinite(bal_acc) else float('nan'),
        'positive_rate_true': float(np.mean(y_true)),
        'positive_rate_pred': float(np.mean(y_pred)),
        'n_samples': int(y_true.shape[0]),
    }

def predictive_equality_ratio_at_threshold(y_true, y_prob, sensitive_df, threshold, attribute_name='age_group'):
    if not isinstance(sensitive_df, pd.DataFrame) or attribute_name not in sensitive_df.columns:
        return float('nan'), pd.DataFrame()

    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)
    attribute_values = sensitive_df[attribute_name].astype(str).fillna('Unknown').to_numpy()

    rows = []
    group_fprs = []
    for group_value in sorted(pd.Series(attribute_values).unique().tolist()):
        mask = attribute_values == group_value
        yt = y_true[mask]
        yp = y_pred[mask]
        negatives = int((yt == 0).sum())
        if negatives == 0:
            fpr_value = float('nan')
        else:
            tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
            fpr_value = safe_div(fp, fp + tn)
        rows.append({
            'attribute_name': attribute_name,
            'attribute_value': group_value,
            'n_samples': int(mask.sum()),
            'n_negatives': negatives,
            'fpr': fpr_value,
        })
        if np.isfinite(fpr_value):
            group_fprs.append(float(fpr_value))

    if len(group_fprs) < 2:
        return float('nan'), pd.DataFrame(rows)

    max_fpr = max(group_fprs)
    min_fpr = min(group_fprs)
    if max_fpr <= 0:
        ratio = 1.0
    else:
        ratio = float(np.clip(min_fpr / max_fpr, 0.0, 1.0))
    return ratio, pd.DataFrame(rows)

def _aequitas_get_crosstabs_df(audit_df: pd.DataFrame, attr_cols):
    group_auditor = AequitasGroup()
    crosstab_result = group_auditor.get_crosstabs(audit_df, attr_cols=attr_cols)
    return crosstab_result[0] if isinstance(crosstab_result, tuple) else crosstab_result

def _aequitas_get_major_group_disparity_df(crosstab_df: pd.DataFrame, audit_df: pd.DataFrame):
    bias_auditor = AequitasBias()
    try:
        disparity_result = bias_auditor.get_disparity_major_group(
            crosstab_df,
            original_df=audit_df,
            mask_significance=False,
        )
    except TypeError:
        try:
            disparity_result = bias_auditor.get_disparity_major_group(
                crosstab_df,
                df=audit_df,
                mask_significance=False,
            )
        except TypeError:
            disparity_result = bias_auditor.get_disparity_major_group(crosstab_df, audit_df)
    return disparity_result[0] if isinstance(disparity_result, tuple) else disparity_result

def aequitas_fpr_parity_summary_at_threshold(y_true, y_probabilities, sensitive_attribute_df, threshold):
    if AequitasGroup is None or AequitasBias is None:
        raise RuntimeError(
            'Aequitas is required for fairness evaluation in this notebook. '
            f'Original import error: {_AEQUITAS_IMPORT_ERROR}'
        )

    y_true_int = np.asarray(y_true).ravel().astype(int)
    y_pred_int = (np.asarray(y_probabilities) >= float(threshold)).astype(int)

    audit_df = sensitive_attribute_df.copy()
    audit_df['score'] = y_pred_int.astype(float)
    audit_df['label_value'] = y_true_int

    attr_cols = [attribute_name for attribute_name in AEQUITAS_FAIRNESS_ATTRIBUTES if attribute_name in audit_df.columns]
    if not attr_cols:
        return float('nan'), {}, pd.DataFrame()

    attribute_group_counts = {}
    for attribute_name in attr_cols:
        attribute_values = audit_df[attribute_name].astype(str).fillna('Unknown')
        attribute_group_counts[attribute_name] = int(pd.Series(attribute_values).nunique(dropna=False))

    crosstab_df = _aequitas_get_crosstabs_df(audit_df, attr_cols=attr_cols)
    disparity_df = _aequitas_get_major_group_disparity_df(crosstab_df, audit_df)

    if isinstance(disparity_df, pd.DataFrame) and not disparity_df.empty and 'attribute_name' in disparity_df.columns:
        disparity_df = disparity_df.copy()
        disparity_df['attribute_n_distinct_groups'] = disparity_df['attribute_name'].map(attribute_group_counts)
        disparity_df['attribute_is_degenerate'] = (
            pd.to_numeric(disparity_df['attribute_n_distinct_groups'], errors='coerce')
            .fillna(0)
            .lt(2)
            .astype(int)
        )

    per_attribute_scores = {}
    for attribute_name in attr_cols:
        group_count = int(attribute_group_counts.get(attribute_name, 0))
        if group_count < 2:
            per_attribute_scores[attribute_name] = float('nan')
            continue

        attribute_rows = disparity_df[disparity_df['attribute_name'] == attribute_name].copy()
        if 'fpr_disparity' not in attribute_rows.columns or attribute_rows.empty:
            per_attribute_scores[attribute_name] = float('nan')
            continue

        disparities = pd.to_numeric(attribute_rows['fpr_disparity'], errors='coerce').to_numpy(dtype=float)
        if disparities.size == 0:
            per_attribute_scores[attribute_name] = float('nan')
            continue

        parity_scores = np.full(disparities.shape, np.nan, dtype=float)
        finite_positive_mask = np.isfinite(disparities) & (disparities > 0.0)
        parity_scores[finite_positive_mask] = np.minimum(
            disparities[finite_positive_mask],
            1.0 / disparities[finite_positive_mask],
        )
        parity_scores[np.isinf(disparities) | (disparities == 0.0)] = 0.0

        valid_parity_scores = parity_scores[np.isfinite(parity_scores)]
        if valid_parity_scores.size == 0:
            per_attribute_scores[attribute_name] = float('nan')
            continue

        per_attribute_scores[attribute_name] = float(np.clip(np.min(valid_parity_scores), 0.0, 1.0))

    finite_scores = [score for score in per_attribute_scores.values() if np.isfinite(score)]
    overall_score = float(min(finite_scores)) if finite_scores else float('nan')
    return overall_score, per_attribute_scores, disparity_df

def aequitas_attribute_metadata_by_attribute(disparity_df: pd.DataFrame):
    if not isinstance(disparity_df, pd.DataFrame) or disparity_df.empty or 'attribute_name' not in disparity_df.columns:
        return {}, {}

    attr_rows = disparity_df[[
        column_name for column_name in [
            'attribute_name',
            'attribute_n_distinct_groups',
            'attribute_is_degenerate',
        ] if column_name in disparity_df.columns
    ]].copy()
    if 'attribute_name' not in attr_rows.columns:
        return {}, {}

    attr_rows = attr_rows.drop_duplicates(subset=['attribute_name'], keep='first')

    if 'attribute_n_distinct_groups' in attr_rows.columns:
        n_distinct_groups_by_attr = {
            str(attribute_name): int(value)
            for attribute_name, value in zip(
                attr_rows['attribute_name'],
                pd.to_numeric(attr_rows['attribute_n_distinct_groups'], errors='coerce').fillna(-1).astype(int),
            )
            if int(value) >= 0
        }
    else:
        n_distinct_groups_by_attr = {}

    if 'attribute_is_degenerate' in attr_rows.columns:
        is_degenerate_by_attr = {
            str(attribute_name): int(value)
            for attribute_name, value in zip(
                attr_rows['attribute_name'],
                pd.to_numeric(attr_rows['attribute_is_degenerate'], errors='coerce').fillna(-1).astype(int),
            )
            if int(value) in (0, 1)
        }
    else:
        is_degenerate_by_attr = {}

    return n_distinct_groups_by_attr, is_degenerate_by_attr


In [ ]:
def ensure_csv_header_compatibility(csv_path: Path, expected_fields):
    if not csv_path.exists():
        return
    with open(csv_path, newline='') as csv_file:
        reader = csv.reader(csv_file)
        existing_header = next(reader, [])
    if list(existing_header) == list(expected_fields):
        return
    archived_path = csv_path.with_suffix(csv_path.suffix + '.legacy')
    csv_path.rename(archived_path)
    print(f'Archived incompatible prior log to {archived_path}')

if RESET_TRIAL_LOG_CSVS:
    print('Resetting trial/best/fairness CSV logs for a fresh export.')
    TRIALS_CSV.unlink(missing_ok=True)
    BEST_CSV.unlink(missing_ok=True)
    FAIR_CSV.unlink(missing_ok=True)
else:
    print('Keeping existing trial/best/fairness CSV logs (resume mode).')
    ensure_csv_header_compatibility(TRIALS_CSV, TRIAL_LOG_FIELDS)
    ensure_csv_header_compatibility(BEST_CSV, TRIAL_LOG_FIELDS)
    ensure_csv_header_compatibility(FAIR_CSV, FAIR_LOG_FIELDS)

def append_csv_row(csv_path: Path, fieldnames, row_dict: dict):
    write_header = not csv_path.exists()
    with open(csv_path, 'a', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)

def lightgbm_fixed_params():
    fixed_params = {
        'objective': 'binary',
        'metric': 'auc',
        'n_estimators': int(MAX_ESTIMATORS),
        'random_state': int(RANDOM_SEED),
        'n_jobs': int(LIGHTGBM_N_JOBS),
        'importance_type': 'gain',
        'class_weight': None,
        'scale_pos_weight': float(scale_pos_weight),
        'verbosity': -1,
        'deterministic': True,
        'force_col_wise': True,
        'data_random_seed': int(RANDOM_SEED),
        'feature_fraction_seed': int(RANDOM_SEED),
        'bagging_seed': int(RANDOM_SEED),
    }
    if LIGHTGBM_DEVICE_TYPE != 'cpu':
        fixed_params['device_type'] = str(LIGHTGBM_DEVICE_TYPE)
        fixed_params['gpu_platform_id'] = int(LIGHTGBM_GPU_PLATFORM_ID)
        fixed_params['gpu_device_id'] = int(LIGHTGBM_GPU_DEVICE_ID)
    return fixed_params

def fit_lightgbm_model(params: dict):
    model = lgb.LGBMClassifier(**params)
    callbacks = [lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)]
    model.fit(
        X_train,
        y_train_np,
        eval_set=[(X_valid, y_valid_np)],
        eval_metric='auc',
        categorical_feature=categorical_cols,
        callbacks=callbacks,
    )
    return model

def predict_probabilities(model, feature_frame: pd.DataFrame):
    best_iteration = getattr(model, 'best_iteration_', None)
    predict_kwargs = {}
    if best_iteration is not None and int(best_iteration) > 0:
        predict_kwargs['num_iteration'] = int(best_iteration)
    return model.predict_proba(feature_frame, **predict_kwargs)[:, 1]

def resolved_best_iteration(model):
    best_iteration = getattr(model, 'best_iteration_', None)
    if best_iteration is None or int(best_iteration) <= 0:
        best_iteration = getattr(model, 'n_estimators_', None)
    if best_iteration is None or int(best_iteration) <= 0:
        best_iteration = model.get_params().get('n_estimators', MAX_ESTIMATORS)
    return int(best_iteration)

def suggest_lightgbm_params(trial: optuna.Trial):
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    return {
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt']),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_categorical('max_depth', [-1, 4, 6, 8, 10, 12]),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 400),
        'subsample': subsample,
        'subsample_freq': 1 if subsample < 0.999 else 0,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 1e-8, 1.0, log=True),
    }

def run_single_trial(trial: optuna.Trial):
    stage_label = 'lightgbm'
    tuned_params = suggest_lightgbm_params(trial)
    model = fit_lightgbm_model({**lightgbm_fixed_params(), **tuned_params})
    best_iteration = resolved_best_iteration(model)

    valid_probs = predict_probabilities(model, X_valid)
    valid_auc = safe_binary_metric(roc_auc_score, y_valid_np, valid_probs)
    valid_prauc = safe_binary_metric(average_precision_score, y_valid_np, valid_probs)
    valid_recall, valid_threshold, valid_fpr = recall_at_fpr_cap_from_scores(y_valid_np, valid_probs, FPR_CAP)

    trial_log_row = {
        'stage': stage_label,
        'trial': trial.number,
        'best_iteration': int(best_iteration),
        **tuned_params,
        'val_auc': valid_auc,
        'val_prauc': valid_prauc,
        'val_recall_at_fpr': valid_recall,
        'val_fpr': valid_fpr,
        'val_threshold': valid_threshold,
    }
    append_csv_row(TRIALS_CSV, TRIAL_LOG_FIELDS, trial_log_row)
    append_csv_row(BEST_CSV, TRIAL_LOG_FIELDS, trial_log_row)

    threshold_star = float(valid_threshold)
    fairness_ratio_valid, fairness_attr_scores_valid, valid_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
        y_valid_np, valid_probs, valid_sensitive_attributes_df, threshold_star
    )
    valid_predictive_equality_age, _ = predictive_equality_ratio_at_threshold(
        y_valid_np, valid_probs, valid_sensitive_attributes_df, threshold_star, attribute_name='age_group'
    )
    valid_attr_n_groups, valid_attr_is_degenerate = aequitas_attribute_metadata_by_attribute(valid_aequitas_disparity_df)

    if LOG_TEST_DURING_TUNING:
        test_probs = predict_probabilities(model, X_test)
        recall_test = confusion_metrics_from_probs(y_test_np, test_probs, threshold_star)['recall']
        fpr_test = confusion_metrics_from_probs(y_test_np, test_probs, threshold_star)['fpr']
        fairness_ratio_test, fairness_attr_scores_test, test_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
            y_test_np, test_probs, test_sensitive_attributes_df, threshold_star
        )
        predictive_equality_age_test, _ = predictive_equality_ratio_at_threshold(
            y_test_np, test_probs, test_sensitive_attributes_df, threshold_star, attribute_name='age_group'
        )
        test_attr_n_groups, test_attr_is_degenerate = aequitas_attribute_metadata_by_attribute(test_aequitas_disparity_df)
    else:
        recall_test = float('nan')
        fpr_test = float('nan')
        fairness_ratio_test = float('nan')
        predictive_equality_age_test = float('nan')
        fairness_attr_scores_test = {}
        test_attr_n_groups = {}
        test_attr_is_degenerate = {}

    fairness_attribute_n_distinct_groups_payload = json.dumps(
        {
            'val': {k: int(v) for k, v in valid_attr_n_groups.items()},
            'test': {k: int(v) for k, v in test_attr_n_groups.items()},
        },
        sort_keys=True,
    )
    fairness_attribute_is_degenerate_payload = json.dumps(
        {
            'val': {k: int(v) for k, v in valid_attr_is_degenerate.items()},
            'test': {k: int(v) for k, v in test_attr_is_degenerate.items()},
        },
        sort_keys=True,
    )

    append_csv_row(
        FAIR_CSV,
        FAIR_LOG_FIELDS,
        {
            'stage': stage_label,
            'trial': trial.number,
            'best_iteration': int(best_iteration),
            'perf_recall_test': recall_test,
            'fairness_ratio_test': fairness_ratio_test,
            'predictive_equality_age_test': predictive_equality_age_test,
            'thr_star': threshold_star,
            'fpr_test': fpr_test,
            'recall_val': valid_recall,
            'fpr_val': valid_fpr,
            'fairness_ratio_val': fairness_ratio_valid,
            'predictive_equality_age_val': valid_predictive_equality_age,
            'aeq_fpr_parity_age_test': fairness_attr_scores_test.get('age_group', float('nan')),
            'aeq_fpr_parity_income_test': fairness_attr_scores_test.get('income_group', float('nan')),
            'aeq_fpr_parity_employment_status_test': fairness_attr_scores_test.get('employment_status_group', float('nan')),
            'aeq_fpr_parity_age_val': fairness_attr_scores_valid.get('age_group', float('nan')),
            'aeq_fpr_parity_income_val': fairness_attr_scores_valid.get('income_group', float('nan')),
            'aeq_fpr_parity_employment_status_val': fairness_attr_scores_valid.get('employment_status_group', float('nan')),
            'attribute_n_distinct_groups': fairness_attribute_n_distinct_groups_payload,
            'attribute_is_degenerate': fairness_attribute_is_degenerate_payload,
        },
    )

    return float(valid_recall)


In [ ]:
def create_tuning_study():
    return optuna.create_study(
        direction=OPTUNA_DIRECTION,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE_URL,
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )

def optimize_to_target(study: optuna.Study, objective_fn, target_trials: int, study_label: str):
    complete_trials = sum(1 for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE)
    failed_trials = sum(1 for trial in study.trials if trial.state == optuna.trial.TrialState.FAIL)
    remaining_trials = max(0, int(target_trials) - int(complete_trials))
    print(f'[{study_label}] completed trials: {complete_trials} | failed trials: {failed_trials} | target: {int(target_trials)} | remaining: {remaining_trials}')
    if remaining_trials > 0:
        study.optimize(objective_fn, n_trials=remaining_trials, show_progress_bar=False)
    else:
        print(f'[{study_label}] completed-trial target already reached; skipping new trials.')
    return remaining_trials

if TUNING_TRIALS <= 0:
    raise ValueError('TUNING_TRIALS must be > 0 to run the LightGBM baseline study.')

print('Running LightGBM tuning study...')
tuning_study = create_tuning_study()
tuning_new_trials = optimize_to_target(tuning_study, run_single_trial, TUNING_TRIALS, 'LightGBM tuning')
best_params = {
    'boosting_type': tuning_study.best_params['boosting_type'],
    'num_leaves': int(tuning_study.best_params['num_leaves']),
    'max_depth': int(tuning_study.best_params['max_depth']),
    'learning_rate': float(tuning_study.best_params['learning_rate']),
    'min_child_samples': int(tuning_study.best_params['min_child_samples']),
    'subsample': float(tuning_study.best_params['subsample']),
    'subsample_freq': 1 if float(tuning_study.best_params['subsample']) < 0.999 else 0,
    'colsample_bytree': float(tuning_study.best_params['colsample_bytree']),
    'reg_alpha': float(tuning_study.best_params['reg_alpha']),
    'reg_lambda': float(tuning_study.best_params['reg_lambda']),
    'min_split_gain': float(tuning_study.best_params['min_split_gain']),
}
final_best_stage = 'lightgbm'
print('[LightGBM tuning] best recall@FPR cap:', tuning_study.best_value)
print('[LightGBM tuning] best params:', best_params)


In [ ]:
final_model = fit_lightgbm_model({**lightgbm_fixed_params(), **best_params})
final_best_iteration = resolved_best_iteration(final_model)

valid_probs = predict_probabilities(final_model, X_valid)
test_probs = predict_probabilities(final_model, X_test)

valid_recall, threshold_star, valid_fpr = recall_at_fpr_cap_from_scores(y_valid_np, valid_probs, FPR_CAP)
threshold_star = float(threshold_star)
valid_metrics_at_thr = confusion_metrics_from_probs(y_valid_np, valid_probs, threshold_star)
test_metrics_at_thr = confusion_metrics_from_probs(y_test_np, test_probs, threshold_star)

valid_auc = float(roc_auc_score(y_valid_np, valid_probs))
valid_prauc = float(average_precision_score(y_valid_np, valid_probs))
valid_pr_auc_trapezoidal = float(trapezoidal_pr_auc_from_scores(y_valid_np, valid_probs))
test_auc = float(roc_auc_score(y_test_np, test_probs))
test_prauc = float(average_precision_score(y_test_np, test_probs))
test_pr_auc_trapezoidal = float(trapezoidal_pr_auc_from_scores(y_test_np, test_probs))

valid_tpr_5, valid_thr_5, valid_fpr_5 = recall_at_fpr_cap_from_scores(y_valid_np, valid_probs, FPR_CAP)
valid_tpr_1, valid_thr_1, valid_fpr_1 = recall_at_fpr_cap_from_scores(y_valid_np, valid_probs, SECONDARY_FPR_CAP)
test_tpr_5, test_thr_5, test_fpr_5 = recall_at_fpr_cap_from_scores(y_test_np, test_probs, FPR_CAP)
test_tpr_1, test_thr_1, test_fpr_1 = recall_at_fpr_cap_from_scores(y_test_np, test_probs, SECONDARY_FPR_CAP)

valid_fairness_overall, valid_fairness_attr_scores, valid_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
    y_valid_np, valid_probs, valid_sensitive_attributes_df, threshold_star
)
test_fairness_overall, test_fairness_attr_scores, test_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
    y_test_np, test_probs, test_sensitive_attributes_df, threshold_star
)
valid_predictive_equality_age_ratio, valid_predictive_equality_age_df = predictive_equality_ratio_at_threshold(
    y_valid_np, valid_probs, valid_sensitive_attributes_df, threshold_star, attribute_name='age_group'
)
test_predictive_equality_age_ratio, test_predictive_equality_age_df = predictive_equality_ratio_at_threshold(
    y_test_np, test_probs, test_sensitive_attributes_df, threshold_star, attribute_name='age_group'
)

print(f'VALID AUC                     : {valid_auc:.6f}')
print(f'VALID Average Precision       : {valid_prauc:.6f}')
print(f'VALID recall@selected_thr     : {valid_metrics_at_thr["recall"]:.6f}')
print(f'VALID selected threshold      : {threshold_star:.6f}')
print(f'VALID splitwise TPR@5%FPR     : {valid_tpr_5:.6f}')
print(f'TEST  AUC                     : {test_auc:.6f}')
print(f'TEST  Average Precision       : {test_prauc:.6f}')
print(f'TEST  recall@selected_thr     : {test_metrics_at_thr["recall"]:.6f}')
print(f'TEST  FPR@selected_thr        : {test_metrics_at_thr["fpr"]:.6f}')
print(f'TEST  splitwise TPR@5%FPR     : {test_tpr_5:.6f}')
print(f'TEST  predictive equality(age): {test_predictive_equality_age_ratio:.6f}')
print(f'Final best iteration          : {final_best_iteration}')

display(pd.DataFrame([best_params]))
display(test_predictive_equality_age_df)


In [ ]:
PAPER_ARTIFACT_DIR = PAPER_EXPORT_DIR
PAPER_TABLES_DIR = PAPER_ARTIFACT_DIR / 'tables'
PAPER_CURVES_DIR = PAPER_ARTIFACT_DIR / 'curves'
PAPER_FIGS_DIR = PAPER_ARTIFACT_DIR / 'figures'
PAPER_PREDS_DIR = PAPER_ARTIFACT_DIR / 'predictions'
for _dir in [PAPER_ARTIFACT_DIR, PAPER_TABLES_DIR, PAPER_CURVES_DIR, PAPER_FIGS_DIR, PAPER_PREDS_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

paper_artifact_registry = []

def register_artifact(path, kind, description):
    path = Path(path)
    paper_artifact_registry.append({
        'kind': str(kind),
        'path': str(path),
        'description': str(description),
        'exists': int(path.exists()),
    })
    return path

def save_df_csv(df: pd.DataFrame, path: Path, description: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    register_artifact(path, 'csv', description)
    return path

def save_fig(fig, path: Path, description: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    register_artifact(path, 'figure', description)
    return path

def make_threshold_sweep_df(y_true, y_prob, selected_threshold, n_points=401):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    thresholds = np.linspace(0.0, 1.0, int(n_points))
    thresholds = np.unique(np.concatenate([thresholds, [float(selected_threshold)]]))
    rows = []
    for thr in thresholds:
        m = confusion_metrics_from_probs(y_true, y_prob, thr)
        rows.append({
            'threshold': float(thr),
            'recall': m['recall'],
            'fpr': m['fpr'],
            'precision': m['precision'],
            'specificity': m['specificity'],
            'f1': m['f1'],
            'positive_rate_pred': m['positive_rate_pred'],
            'within_fpr_cap': int(np.isfinite(m['fpr']) and m['fpr'] <= FPR_CAP + 1e-12),
            'is_selected_threshold': int(abs(float(thr) - float(selected_threshold)) < 1e-12),
        })
    sweep_df = pd.DataFrame(rows).sort_values('threshold').reset_index(drop=True)
    if not sweep_df.empty:
        under_cap = sweep_df[sweep_df['within_fpr_cap'] == 1]
        if not under_cap.empty and under_cap['recall'].notna().any():
            best_idx = under_cap['recall'].astype(float).idxmax()
            sweep_df['is_best_recall_under_cap'] = 0
            sweep_df.loc[best_idx, 'is_best_recall_under_cap'] = 1
        else:
            sweep_df['is_best_recall_under_cap'] = 0
    return sweep_df

def make_roc_df(y_true, y_prob):
    fpr, tpr, thr = roc_curve(np.asarray(y_true).astype(int).ravel(), np.asarray(y_prob, dtype=float).ravel())
    return pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thr})

def make_pr_df(y_true, y_prob):
    precision, recall, thr = precision_recall_curve(np.asarray(y_true).astype(int).ravel(), np.asarray(y_prob, dtype=float).ravel())
    threshold_series = np.full(shape=(len(precision),), fill_value=np.nan, dtype=float)
    threshold_series[:len(thr)] = thr
    return pd.DataFrame({'precision': precision, 'recall': recall, 'threshold': threshold_series})

def make_calibration_df(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    try:
        frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=int(n_bins), strategy='quantile')
        calib_df = pd.DataFrame({'mean_predicted_prob': mean_pred, 'fraction_positives': frac_pos})
    except Exception:
        calib_df = pd.DataFrame(columns=['mean_predicted_prob', 'fraction_positives'])
    calib_df['n_bins_requested'] = int(n_bins)
    return calib_df

def build_prediction_export_df(y_true, y_prob, threshold, sensitive_df, split_label):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)
    export_df = pd.DataFrame({
        'split': str(split_label),
        'row_idx': np.arange(len(y_true), dtype=int),
        'y_true': y_true,
        'y_prob': y_prob,
        'y_pred_at_selected_threshold': y_pred,
    })
    if isinstance(sensitive_df, pd.DataFrame):
        for col in ['age_group', 'income_group', 'employment_status_group']:
            if col in sensitive_df.columns and len(sensitive_df) == len(export_df):
                export_df[col] = sensitive_df[col].astype(str).to_numpy()
    return export_df

def build_subgroup_metrics_df(y_true, y_prob, threshold, sensitive_df, split_label):
    rows = []
    if not isinstance(sensitive_df, pd.DataFrame) or sensitive_df.empty:
        return pd.DataFrame(rows)

    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)

    for attr in [a for a in AEQUITAS_FAIRNESS_ATTRIBUTES if a in sensitive_df.columns]:
        values = sensitive_df[attr].astype(str).fillna('Unknown')
        for group_value in sorted(values.unique().tolist()):
            mask = (values == group_value).to_numpy()
            if int(mask.sum()) == 0:
                continue
            yt = y_true[mask]
            yp = y_prob[mask]
            yhat = y_pred[mask]
            tn, fp, fn, tp = confusion_matrix(yt, yhat, labels=[0, 1]).ravel()
            precision = safe_div(tp, tp + fp)
            recall = safe_div(tp, tp + fn)
            rows.append({
                'split': str(split_label),
                'attribute_name': str(attr),
                'attribute_value': str(group_value),
                'n_samples': int(mask.sum()),
                'positive_rate_true': float(np.mean(yt)),
                'positive_rate_pred': float(np.mean(yhat)),
                'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
                'precision': precision,
                'recall': recall,
                'fpr': safe_div(fp, fp + tn),
                'specificity': safe_div(tn, tn + fp),
                'f1': safe_div(2 * precision * recall, precision + recall) if np.isfinite(precision) and np.isfinite(recall) else float('nan'),
                'auc': safe_binary_metric(roc_auc_score, yt, yp),
                'pr_auc': safe_binary_metric(average_precision_score, yt, yp),
            })
    return pd.DataFrame(rows)

def summarize_split_frame(df: pd.DataFrame, split_label: str):
    if df is None or len(df) == 0:
        return {'split': split_label, 'n_samples': 0}
    month_min = int(df[MONTH_COLUMN].min()) if MONTH_COLUMN in df.columns else None
    month_max = int(df[MONTH_COLUMN].max()) if MONTH_COLUMN in df.columns else None
    target_rate = float(df[target_col].mean()) if target_col in df.columns else float('nan')
    return {
        'split': str(split_label),
        'n_samples': int(len(df)),
        'n_features_total': int(len(feature_cols)),
        'n_categorical_features': int(len(categorical_cols)),
        'n_continuous_features': int(len(continuous_cols)),
        'positive_rate': target_rate,
        'positive_count': int(df[target_col].sum()) if target_col in df.columns else None,
        'negative_count': int((1 - df[target_col]).sum()) if target_col in df.columns else None,
        'month_min': month_min,
        'month_max': month_max,
    }

def prepare_aequitas_export_df(disparity_df: pd.DataFrame, split_label: str):
    if disparity_df is None or len(disparity_df) == 0:
        return pd.DataFrame()
    out = disparity_df.copy()
    out.insert(0, 'split', str(split_label))
    if 'attribute_name' in out.columns:
        out = out[out['attribute_name'].isin(AEQUITAS_FAIRNESS_ATTRIBUTES)].copy()
    preferred_cols = [
        'split', 'attribute_name', 'attribute_value', 'attribute_n_distinct_groups', 'attribute_is_degenerate', 'label_value', 'score',
        'fpr', 'fpr_disparity', 'tpr', 'tpr_disparity', 'tnr', 'tnr_disparity',
        'fnr', 'fnr_disparity', 'pprev', 'ppr', 'ppr_disparity'
    ]
    keep = [c for c in preferred_cols if c in out.columns]
    extras = [c for c in out.columns if c not in keep]
    return out[keep + extras]

def build_attr_parity_summary_df(valid_scores: dict, test_scores: dict):
    attrs = sorted(set(list(valid_scores.keys()) + list(test_scores.keys())))
    rows = []
    for attr in attrs:
        rows.append({
            'attribute_name': attr,
            'valid_fpr_parity_score': float(valid_scores.get(attr, np.nan)),
            'test_fpr_parity_score': float(test_scores.get(attr, np.nan)),
        })
    return pd.DataFrame(rows)

trials_df = pd.read_csv(TRIALS_CSV)
best_df = pd.read_csv(BEST_CSV)
fair_df = pd.read_csv(FAIR_CSV)

final_metrics_df = pd.DataFrame([
    {
        'split': 'VALID',
        'auc': valid_auc,
        'pr_auc': valid_prauc,
        'pr_average_precision_score': valid_prauc,
        'pr_auc_trapezoidal': valid_pr_auc_trapezoidal,
        'pr_dylan_snippet_proxy': dylan_snippet_pr_score_from_point(valid_metrics_at_thr['precision'], valid_metrics_at_thr['recall']),
        'recall_at_selected_threshold': valid_metrics_at_thr['recall'],
        'fpr_at_selected_threshold': valid_metrics_at_thr['fpr'],
        'precision_at_selected_threshold': valid_metrics_at_thr['precision'],
        'f1_at_selected_threshold': valid_metrics_at_thr['f1'],
        'balanced_accuracy_at_selected_threshold': valid_metrics_at_thr['balanced_accuracy'],
        'selected_threshold': threshold_star,
        'fpr_cap': float(FPR_CAP),
        'tpr_at_5pct_fpr_splitwise': valid_tpr_5,
        'tpr_at_1pct_fpr_splitwise': valid_tpr_1,
        'predictive_equality_age_ratio': valid_predictive_equality_age_ratio,
        'aequitas_fpr_parity_overall': float(valid_fairness_overall),
        'positive_rate_true': valid_metrics_at_thr['positive_rate_true'],
        'positive_rate_pred': valid_metrics_at_thr['positive_rate_pred'],
        'n_samples': valid_metrics_at_thr['n_samples'],
    },
    {
        'split': 'TEST',
        'auc': test_auc,
        'pr_auc': test_prauc,
        'pr_average_precision_score': test_prauc,
        'pr_auc_trapezoidal': test_pr_auc_trapezoidal,
        'pr_dylan_snippet_proxy': dylan_snippet_pr_score_from_point(test_metrics_at_thr['precision'], test_metrics_at_thr['recall']),
        'recall_at_selected_threshold': test_metrics_at_thr['recall'],
        'fpr_at_selected_threshold': test_metrics_at_thr['fpr'],
        'precision_at_selected_threshold': test_metrics_at_thr['precision'],
        'f1_at_selected_threshold': test_metrics_at_thr['f1'],
        'balanced_accuracy_at_selected_threshold': test_metrics_at_thr['balanced_accuracy'],
        'selected_threshold': threshold_star,
        'fpr_cap': float(FPR_CAP),
        'tpr_at_5pct_fpr_splitwise': test_tpr_5,
        'tpr_at_1pct_fpr_splitwise': test_tpr_1,
        'predictive_equality_age_ratio': test_predictive_equality_age_ratio,
        'aequitas_fpr_parity_overall': float(test_fairness_overall),
        'positive_rate_true': test_metrics_at_thr['positive_rate_true'],
        'positive_rate_pred': test_metrics_at_thr['positive_rate_pred'],
        'n_samples': test_metrics_at_thr['n_samples'],
    },
])
save_df_csv(final_metrics_df, PAPER_TABLES_DIR / 'paper_table_final_metrics.csv', 'Final VALID/TEST performance + fairness summary at selected threshold')

baf_headline_df = final_metrics_df[[
    'split',
    'auc',
    'pr_average_precision_score',
    'balanced_accuracy_at_selected_threshold',
    'tpr_at_5pct_fpr_splitwise',
    'tpr_at_1pct_fpr_splitwise',
    'predictive_equality_age_ratio',
    'selected_threshold',
    'recall_at_selected_threshold',
    'fpr_at_selected_threshold',
]].copy()
save_df_csv(baf_headline_df, PAPER_TABLES_DIR / 'paper_table_baf_headline_metrics.csv', 'Reviewer-facing BAF headline metrics (VALID/TEST)')

confusion_summary_df = pd.DataFrame([
    {'split': 'VALID', 'threshold': threshold_star, **valid_metrics_at_thr},
    {'split': 'TEST', 'threshold': threshold_star, **test_metrics_at_thr},
])
save_df_csv(confusion_summary_df, PAPER_TABLES_DIR / 'paper_table_confusion_summary.csv', 'Confusion matrix counts and derived metrics at selected threshold')

pr_metric_comparison_df = pd.DataFrame([
    {
        'split': 'VALID',
        'selected_threshold': threshold_star,
        'precision_at_selected_threshold': valid_metrics_at_thr['precision'],
        'recall_at_selected_threshold': valid_metrics_at_thr['recall'],
        'pr_average_precision_score': valid_prauc,
        'pr_auc_trapezoidal': valid_pr_auc_trapezoidal,
        'pr_dylan_snippet_auc': dylan_snippet_pr_score_from_point(valid_metrics_at_thr['precision'], valid_metrics_at_thr['recall']),
    },
    {
        'split': 'TEST',
        'selected_threshold': threshold_star,
        'precision_at_selected_threshold': test_metrics_at_thr['precision'],
        'recall_at_selected_threshold': test_metrics_at_thr['recall'],
        'pr_average_precision_score': test_prauc,
        'pr_auc_trapezoidal': test_pr_auc_trapezoidal,
        'pr_dylan_snippet_auc': dylan_snippet_pr_score_from_point(test_metrics_at_thr['precision'], test_metrics_at_thr['recall']),
    },
])
save_df_csv(pr_metric_comparison_df, PAPER_TABLES_DIR / 'paper_table_pr_metric_comparison.csv', 'PR summary comparison: average_precision_score, trapezoidal PR-AUC, and Dylan snippet single-point proxy')

split_summary_df = pd.DataFrame([
    summarize_split_frame(train_df, 'TRAIN'),
    summarize_split_frame(valid_df, 'VALID'),
    summarize_split_frame(test_df, 'TEST'),
])
save_df_csv(split_summary_df, PAPER_TABLES_DIR / 'paper_table_dataset_split_summary.csv', 'Dataset split summary (counts, prevalence, month ranges)')

feature_summary_df = pd.DataFrame([
    {'table': 'summary', 'n_total_features': len(feature_cols), 'n_categorical_features': len(categorical_cols), 'n_continuous_features': len(continuous_cols)}
])
save_df_csv(feature_summary_df, PAPER_TABLES_DIR / 'paper_table_feature_summary.csv', 'Feature-type summary counts')

feature_inventory_df = pd.DataFrame({
    'feature_name': feature_cols,
    'inferred_type': ['categorical' if c in categorical_cols else 'continuous' for c in feature_cols],
    'dtype': [str(full_df[c].dtype) for c in feature_cols],
    'n_unique': [int(full_df[c].nunique(dropna=False)) for c in feature_cols],
})
save_df_csv(feature_inventory_df, PAPER_TABLES_DIR / 'paper_table_feature_inventory.csv', 'Feature inventory for appendix/supplement')

best_params_export_df = pd.DataFrame([
    {
        'experiment_tag': EXPERIMENT_TAG,
        'dataset_csv': str(DATA_CSV_PATH),
        'selected_stage': final_best_stage,
        'selected_threshold': threshold_star,
        'selected_threshold_fpr_cap': float(FPR_CAP),
        'best_iteration': int(final_best_iteration),
        'scale_pos_weight': float(scale_pos_weight),
        **best_params,
    }
])
save_df_csv(best_params_export_df, PAPER_TABLES_DIR / 'paper_table_selected_params.csv', 'Selected LightGBM hyperparameters and operating threshold')

merged_trial_summary_df = best_df.merge(
    fair_df,
    on=['stage', 'trial', 'best_iteration'],
    how='left',
    suffixes=('_best', '_fair')
)
save_df_csv(merged_trial_summary_df, PAPER_TABLES_DIR / 'paper_table_trials_best_plus_fairness.csv', 'Merged best-per-trial metrics with fairness summary')

stage_summary_df = pd.DataFrame([
    {
        'stage': 'lightgbm',
        'n_trials_best_rows': int(len(best_df)),
        'val_recall_at_fpr_mean': float(pd.to_numeric(best_df['val_recall_at_fpr'], errors='coerce').mean()),
        'val_recall_at_fpr_max': float(pd.to_numeric(best_df['val_recall_at_fpr'], errors='coerce').max()),
        'val_auc_mean': float(pd.to_numeric(best_df['val_auc'], errors='coerce').mean()),
        'val_prauc_mean': float(pd.to_numeric(best_df['val_prauc'], errors='coerce').mean()),
        'fairness_ratio_val_mean': float(pd.to_numeric(merged_trial_summary_df.get('fairness_ratio_val', pd.Series(dtype=float)), errors='coerce').mean()),
        'fairness_ratio_test_mean': float(pd.to_numeric(merged_trial_summary_df.get('fairness_ratio_test', pd.Series(dtype=float)), errors='coerce').mean()),
        'predictive_equality_age_val_mean': float(pd.to_numeric(merged_trial_summary_df.get('predictive_equality_age_val', pd.Series(dtype=float)), errors='coerce').mean()),
        'predictive_equality_age_test_mean': float(pd.to_numeric(merged_trial_summary_df.get('predictive_equality_age_test', pd.Series(dtype=float)), errors='coerce').mean()),
    }
])
save_df_csv(stage_summary_df, PAPER_TABLES_DIR / 'paper_table_stage_summary.csv', 'Stage-wise tuning summary statistics')

top_trials_export_df = merged_trial_summary_df.sort_values('val_recall_at_fpr', ascending=False).head(max(10, TOP_K_TRIALS)).copy()
save_df_csv(top_trials_export_df, PAPER_TABLES_DIR / 'paper_table_top_trials.csv', 'Top trials by validation recall@FPR cap with fairness and hyperparameters')

valid_predictions_df = build_prediction_export_df(y_valid_np, valid_probs, threshold_star, valid_sensitive_attributes_df, 'VALID')
test_predictions_df = build_prediction_export_df(y_test_np, test_probs, threshold_star, test_sensitive_attributes_df, 'TEST')
save_df_csv(valid_predictions_df, PAPER_PREDS_DIR / 'predictions_valid.csv', 'Per-row VALID predictions at selected threshold')
save_df_csv(test_predictions_df, PAPER_PREDS_DIR / 'predictions_test.csv', 'Per-row TEST predictions at selected threshold')

valid_roc_df = make_roc_df(y_valid_np, valid_probs)
test_roc_df = make_roc_df(y_test_np, test_probs)
valid_pr_df = make_pr_df(y_valid_np, valid_probs)
test_pr_df = make_pr_df(y_test_np, test_probs)
valid_threshold_sweep_df = make_threshold_sweep_df(y_valid_np, valid_probs, threshold_star)
test_threshold_sweep_df = make_threshold_sweep_df(y_test_np, test_probs, threshold_star)
valid_calibration_df = make_calibration_df(y_valid_np, valid_probs, n_bins=10)
test_calibration_df = make_calibration_df(y_test_np, test_probs, n_bins=10)

save_df_csv(valid_roc_df, PAPER_CURVES_DIR / 'curve_roc_valid.csv', 'ROC curve points (VALID)')
save_df_csv(test_roc_df, PAPER_CURVES_DIR / 'curve_roc_test.csv', 'ROC curve points (TEST)')
save_df_csv(valid_pr_df, PAPER_CURVES_DIR / 'curve_pr_valid.csv', 'PR curve points (VALID)')
save_df_csv(test_pr_df, PAPER_CURVES_DIR / 'curve_pr_test.csv', 'PR curve points (TEST)')
save_df_csv(valid_threshold_sweep_df, PAPER_CURVES_DIR / 'curve_threshold_sweep_valid.csv', 'Threshold sweep metrics (VALID)')
save_df_csv(test_threshold_sweep_df, PAPER_CURVES_DIR / 'curve_threshold_sweep_test.csv', 'Threshold sweep metrics (TEST)')
save_df_csv(valid_calibration_df, PAPER_CURVES_DIR / 'curve_calibration_valid.csv', 'Calibration bins (VALID)')
save_df_csv(test_calibration_df, PAPER_CURVES_DIR / 'curve_calibration_test.csv', 'Calibration bins (TEST)')

valid_aequitas_export_df = prepare_aequitas_export_df(valid_aequitas_disparity_df, 'VALID')
test_aequitas_export_df = prepare_aequitas_export_df(test_aequitas_disparity_df, 'TEST')
save_df_csv(valid_aequitas_export_df, PAPER_TABLES_DIR / 'paper_table_aequitas_disparity_valid.csv', 'Aequitas disparity table (VALID, tracked attributes)')
save_df_csv(test_aequitas_export_df, PAPER_TABLES_DIR / 'paper_table_aequitas_disparity_test.csv', 'Aequitas disparity table (TEST, tracked attributes)')

attr_parity_summary_df = build_attr_parity_summary_df(valid_fairness_attr_scores, test_fairness_attr_scores)
save_df_csv(attr_parity_summary_df, PAPER_TABLES_DIR / 'paper_table_fairness_attr_parity_summary.csv', 'Aequitas FPR parity summary by attribute (VALID/TEST)')

valid_subgroup_df = build_subgroup_metrics_df(y_valid_np, valid_probs, threshold_star, valid_sensitive_attributes_df, 'VALID')
test_subgroup_df = build_subgroup_metrics_df(y_test_np, test_probs, threshold_star, test_sensitive_attributes_df, 'TEST')
save_df_csv(valid_subgroup_df, PAPER_TABLES_DIR / 'paper_table_subgroup_metrics_valid.csv', 'Subgroup metrics at selected threshold (VALID)')
save_df_csv(test_subgroup_df, PAPER_TABLES_DIR / 'paper_table_subgroup_metrics_test.csv', 'Subgroup metrics at selected threshold (TEST)')

predictive_equality_df = pd.concat([
    valid_predictive_equality_age_df.assign(split='VALID'),
    test_predictive_equality_age_df.assign(split='TEST'),
], ignore_index=True)
save_df_csv(predictive_equality_df, PAPER_TABLES_DIR / 'paper_table_predictive_equality_age.csv', 'Age-group predictive-equality details at selected threshold')

final_metrics_summary = {
    'valid_auc': float(valid_auc),
    'valid_average_precision': float(valid_prauc),
    'valid_recall_at_selected_threshold': float(valid_metrics_at_thr['recall']),
    'valid_fpr_at_selected_threshold': float(valid_metrics_at_thr['fpr']),
    'valid_tpr_at_5pct_fpr_splitwise': float(valid_tpr_5),
    'test_auc': float(test_auc),
    'test_average_precision': float(test_prauc),
    'test_recall_at_selected_threshold': float(test_metrics_at_thr['recall']),
    'test_fpr_at_selected_threshold': float(test_metrics_at_thr['fpr']),
    'test_tpr_at_5pct_fpr_splitwise': float(test_tpr_5),
    'test_predictive_equality_age_ratio': float(test_predictive_equality_age_ratio) if np.isfinite(test_predictive_equality_age_ratio) else float('nan'),
    'threshold_star': float(threshold_star),
    'valid_aequitas_fpr_parity_overall': float(valid_fairness_overall),
    'test_aequitas_fpr_parity_overall': float(test_fairness_overall),
}

joblib.dump(final_model, MODEL_EXPORT_DIR / 'model.joblib')
register_artifact(MODEL_EXPORT_DIR / 'model.joblib', 'binary', 'Serialized LightGBM model')
with open(MODEL_EXPORT_DIR / 'meta.json', 'w') as meta_file:
    json.dump({
        'experiment_tag': EXPERIMENT_TAG,
        'dataset_csv': str(DATA_CSV_PATH),
        'target': target_col,
        'categorical_cols': categorical_cols,
        'continuous_cols': continuous_cols,
        'fpr_cap': FPR_CAP,
        'secondary_fpr_cap': SECONDARY_FPR_CAP,
        'threshold': float(threshold_star),
        'selected_stage': final_best_stage,
        'optuna_study_name': OPTUNA_STUDY_NAME,
        'tuning_best_params': best_params,
        'best_iteration': int(final_best_iteration),
        'final_metrics_summary': final_metrics_summary,
    }, meta_file, indent=2)
register_artifact(MODEL_EXPORT_DIR / 'meta.json', 'json', 'Run metadata and final metric summary')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=PLOT_DPI)
axes[0].plot(valid_roc_df['fpr'], valid_roc_df['tpr'], label=f'VALID AUC={valid_auc:.3f}')
axes[0].plot(test_roc_df['fpr'], test_roc_df['tpr'], label=f'TEST AUC={test_auc:.3f}')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

axes[1].plot(valid_pr_df['recall'], valid_pr_df['precision'], label=f'VALID AP={valid_prauc:.3f}')
axes[1].plot(test_pr_df['recall'], test_pr_df['precision'], label=f'TEST AP={test_prauc:.3f}')
axes[1].scatter([valid_metrics_at_thr['recall']], [valid_metrics_at_thr['precision']], marker='o', s=35)
axes[1].scatter([test_metrics_at_thr['recall']], [test_metrics_at_thr['precision']], marker='o', s=35)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves')
axes[1].legend()
save_fig(fig, PAPER_FIGS_DIR / 'fig_roc_pr_curves_valid_test.png', 'ROC and PR curves (VALID vs TEST) with operating point markers')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=PLOT_DPI)
for metric_name in ['recall', 'fpr', 'precision']:
    axes[0].plot(valid_threshold_sweep_df['threshold'], valid_threshold_sweep_df[metric_name], label=metric_name)
    axes[1].plot(test_threshold_sweep_df['threshold'], test_threshold_sweep_df[metric_name], label=metric_name)
axes[0].axvline(threshold_star, linestyle='--', color='black', linewidth=1)
axes[1].axvline(threshold_star, linestyle='--', color='black', linewidth=1)
axes[0].set_title('VALID Threshold Sweep')
axes[1].set_title('TEST Threshold Sweep')
for ax in axes:
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Metric value')
    ax.legend()
save_fig(fig, PAPER_FIGS_DIR / 'fig_threshold_sweeps_valid_test.png', 'Threshold sweep curves (Recall/FPR/Precision) for VALID and TEST')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=PLOT_DPI)
axes[0].hist(valid_probs[y_valid_np == 0], bins=40, alpha=0.6, label='negative')
axes[0].hist(valid_probs[y_valid_np == 1], bins=40, alpha=0.6, label='positive')
axes[0].axvline(threshold_star, linestyle='--', color='black', linewidth=1)
axes[0].set_title('VALID Score Distribution')
axes[1].hist(test_probs[y_test_np == 0], bins=40, alpha=0.6, label='negative')
axes[1].hist(test_probs[y_test_np == 1], bins=40, alpha=0.6, label='positive')
axes[1].axvline(threshold_star, linestyle='--', color='black', linewidth=1)
axes[1].set_title('TEST Score Distribution')
for ax in axes:
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Count')
    ax.legend()
save_fig(fig, PAPER_FIGS_DIR / 'fig_score_distributions_valid_test.png', 'Predicted score distributions by class for VALID and TEST')

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
if not attr_parity_summary_df.empty:
    x = np.arange(len(attr_parity_summary_df))
    width = 0.35
    ax.bar(x - width / 2, attr_parity_summary_df['valid_fpr_parity_score'], width=width, label='VALID')
    ax.bar(x + width / 2, attr_parity_summary_df['test_fpr_parity_score'], width=width, label='TEST')
    ax.set_xticks(x)
    ax.set_xticklabels(attr_parity_summary_df['attribute_name'], rotation=20)
ax.set_ylim(0, 1.05)
ax.set_ylabel('FPR parity score')
ax.set_title('Aequitas FPR Parity by Attribute')
ax.legend()
save_fig(fig, PAPER_FIGS_DIR / 'fig_fairness_attr_parity_summary.png', 'Attribute-level Aequitas FPR parity summary for VALID and TEST')

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
scatter_df = fair_df.copy()
if LOG_TEST_DURING_TUNING and scatter_df['perf_recall_test'].notna().any():
    perf_col = 'perf_recall_test'
    fair_col = 'fairness_ratio_test'
    split_label = 'TEST'
else:
    perf_col = 'recall_val'
    fair_col = 'fairness_ratio_val'
    split_label = 'VALID'
ax.scatter(scatter_df[perf_col], scatter_df[fair_col], alpha=0.7)
top_trials_by_recall = scatter_df.sort_values(perf_col, ascending=False).head(TOP_K_TRIALS)
ax.scatter(top_trials_by_recall[perf_col], top_trials_by_recall[fair_col], s=70, edgecolors='black')
ax.set_xlabel(f'Performance (Recall @ {int(FPR_CAP * 100)}% FPR) - {split_label}')
ax.set_ylabel(f'Fairness (Aequitas FPR parity summary) - {split_label} (ideal = 1)')
ax.set_title('Aequitas Fairness vs Performance')
save_fig(fig, PAPER_FIGS_DIR / 'fig_stage_tradeoff_scatter_annotated.png', 'Annotated performance-fairness tradeoff scatter across LightGBM trials')

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
progress_df = best_df.sort_values('trial').copy()
progress_df['cumulative_best_recall'] = pd.to_numeric(progress_df['val_recall_at_fpr'], errors='coerce').cummax()
ax.plot(progress_df['trial'], progress_df['cumulative_best_recall'], marker='o')
ax.set_xlabel('Trial')
ax.set_ylabel(f'Cumulative best recall @ {int(FPR_CAP * 100)}% FPR')
ax.set_title('LightGBM Tuning Progress')
save_fig(fig, PAPER_FIGS_DIR / 'fig_stage_cumulative_best_progress.png', 'Cumulative best recall progression across LightGBM tuning trials')

paper_artifact_manifest_df = pd.DataFrame(paper_artifact_registry)
if paper_artifact_manifest_df.empty:
    raise RuntimeError('No paper artifacts were registered.')
paper_artifact_manifest_df['exists'] = paper_artifact_manifest_df['path'].map(lambda p: int(Path(p).exists()))
paper_artifact_manifest_df = paper_artifact_manifest_df.sort_values(['kind', 'path']).reset_index(drop=True)
save_df_csv(paper_artifact_manifest_df, PAPER_ARTIFACT_DIR / 'paper_artifact_manifest.csv', 'Manifest of generated paper CSVs and figures')
with open(PAPER_ARTIFACT_DIR / 'paper_artifact_manifest.json', 'w') as f:
    json.dump(paper_artifact_manifest_df.to_dict(orient='records'), f, indent=2)
register_artifact(PAPER_ARTIFACT_DIR / 'paper_artifact_manifest.json', 'json', 'Manifest of generated paper artifacts (JSON)')

checklist_entries = [
    ('table_final_metrics', PAPER_TABLES_DIR / 'paper_table_final_metrics.csv'),
    ('table_baf_headline', PAPER_TABLES_DIR / 'paper_table_baf_headline_metrics.csv'),
    ('table_split_summary', PAPER_TABLES_DIR / 'paper_table_dataset_split_summary.csv'),
    ('table_selected_params', PAPER_TABLES_DIR / 'paper_table_selected_params.csv'),
    ('table_stage_summary', PAPER_TABLES_DIR / 'paper_table_stage_summary.csv'),
    ('table_top_trials', PAPER_TABLES_DIR / 'paper_table_top_trials.csv'),
    ('table_fairness_attr', PAPER_TABLES_DIR / 'paper_table_fairness_attr_parity_summary.csv'),
    ('curve_roc_valid', PAPER_CURVES_DIR / 'curve_roc_valid.csv'),
    ('curve_roc_test', PAPER_CURVES_DIR / 'curve_roc_test.csv'),
    ('curve_pr_valid', PAPER_CURVES_DIR / 'curve_pr_valid.csv'),
    ('curve_pr_test', PAPER_CURVES_DIR / 'curve_pr_test.csv'),
    ('curve_threshold_valid', PAPER_CURVES_DIR / 'curve_threshold_sweep_valid.csv'),
    ('curve_threshold_test', PAPER_CURVES_DIR / 'curve_threshold_sweep_test.csv'),
    ('fig_roc_pr', PAPER_FIGS_DIR / 'fig_roc_pr_curves_valid_test.png'),
    ('fig_threshold_sweeps', PAPER_FIGS_DIR / 'fig_threshold_sweeps_valid_test.png'),
    ('fig_score_distributions', PAPER_FIGS_DIR / 'fig_score_distributions_valid_test.png'),
    ('fig_fairness_parity', PAPER_FIGS_DIR / 'fig_fairness_attr_parity_summary.png'),
    ('fig_tradeoff_scatter', PAPER_FIGS_DIR / 'fig_stage_tradeoff_scatter_annotated.png'),
    ('fig_tuning_progress', PAPER_FIGS_DIR / 'fig_stage_cumulative_best_progress.png'),
]
paper_artifact_checklist_df = pd.DataFrame([
    {'artifact_key': key, 'path': str(path), 'exists': int(Path(path).exists())}
    for key, path in checklist_entries
])
save_df_csv(paper_artifact_checklist_df, PAPER_ARTIFACT_DIR / 'paper_artifact_checklist.csv', 'Checklist of core LightGBM paper artifacts')

print(f'Saved model + meta to {MODEL_EXPORT_DIR}/')
print(f'CSV export dir       : {CSV_EXPORT_DIR}')
print(f'Paper artifact dir   : {PAPER_ARTIFACT_DIR}')
print(f'Paper manifest rows  : {len(paper_artifact_manifest_df)}')
print(f'LightGBM notebook run complete for {EXPERIMENT_TAG}.')
